In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# Cycle time in minutes
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
BASE_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
EXTRA_MACHINE = "TOYO 80T 2ND"
ALL_TARGET_MACHINES = BASE_120T_MACHINES | {EXTRA_MACHINE}

AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# BUILD PART → MACHINE MAP
# =============================
records = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = ",".join(grp["Vertical Machines"].astype(str))
    tokens = re.split(r"[,\|/\\\n]+", raw)

    machines = set()
    for t in tokens:
        m = normalize_machine(t)
        if m in BASE_120T_MACHINES:
            machines.add(m)

    # 🔥 If part is 120T-capable, add TOYO 80T 2ND
    if machines:
        machines.add(EXTRA_MACHINE)

    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Possible Machines": ", ".join(sorted(machines))
    })

part_machine_df = pd.DataFrame(records)

# =============================
# BUILD PART-WISE CAPACITY TABLE
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in row["Possible Machines"].split(", "):
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== PART → POSSIBLE MACHINES ==========\n")
display(part_machine_df)

print("\n========== PART-WISE MACHINE CAPACITY (1 DAY) ==========\n")
display(capacity_df)


In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"
output_file = "part_machine_capacity.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")

cycle_time_min = (
    ppm.dropna(subset=["Material", "Machine"])
       .set_index("Material")["Machine"]
       .div(60)   # seconds → minutes
       .to_dict()
)

# =============================
# MACHINE GROUP (INTERCHANGEABLE)
# =============================
MACHINE_GROUP = [
    "MP-01", "MP-05", "MP-10", "MP-17", "TOYO 80T 2ND"
]

AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: PART → MACHINE MAP (ALL PARTS)
# =============================
part_machine_rows = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = grp["Vertical Machines"].dropna()
    if raw.empty:
        continue  # skip only if no machine info at all

    tokens = re.split(r"[,\|/\\\n]+", ",".join(raw.astype(str)))
    normalized = {normalize_machine(t) for t in tokens if normalize_machine(t)}

    # If part appears on ANY machine of the group → assign ALL machines
    if normalized.intersection(MACHINE_GROUP):
        part_machine_rows.append({
            "Child Part": child,
            "Possible Machines": ", ".join(MACHINE_GROUP)
        })

part_machine_df = pd.DataFrame(part_machine_rows)

# =============================
# STEP 2: PART-WISE MACHINE CAPACITY
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in MACHINE_GROUP:
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# WRITE TO EXCEL (2 SHEETS)
# =============================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    part_machine_df.to_excel(writer, sheet_name="Part_Machine_Map", index=False)
    capacity_df.to_excel(writer, sheet_name="Part_Machine_Capacity", index=False)

print(f"✅ Excel file created: {output_file}")


In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"
output_file = "part_machine_capacity_120T.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")

# Cycle time in minutes (seconds → minutes)
cycle_time_min = (
    ppm.dropna(subset=["Material", "Machine"])
       .set_index("Material")["Machine"]
       .div(60)
       .to_dict()
)

# =============================
# CONSTANTS (120T ONLY)
# =============================
MACHINE_GROUP_120T = ["MP-01", "MP-05", "MP-10", "MP-17"]
AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: PART → MACHINE MAP (120T)
# =============================
part_machine_rows = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = grp["Vertical Machines"].dropna()
    if raw.empty:
        continue

    tokens = re.split(r"[,\|/\\\n]+", ",".join(raw.astype(str)))
    normalized = {normalize_machine(t) for t in tokens if normalize_machine(t)}

    # If part appears on ANY 120T machine → allow ALL 120T machines
    if normalized.intersection(MACHINE_GROUP_120T):
        part_machine_rows.append({
            "Child Part": child,
            "Possible Machines": ", ".join(MACHINE_GROUP_120T)
        })

part_machine_df = pd.DataFrame(part_machine_rows)

# =============================
# STEP 2: PART-WISE MACHINE CAPACITY (120T)
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in MACHINE_GROUP_120T:
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# WRITE TO EXCEL (2 SHEETS)
# =============================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    part_machine_df.to_excel(writer, sheet_name="Part_Machine_Map", index=False)
    capacity_df.to_excel(writer, sheet_name="Part_Machine_Capacity", index=False)

print(f"✅ Excel file created: {output_file}")
